# Data Cleaning & Processed Output

## Key Data Engineering Finding
The 3 datasets use **incompatible ID schemes** (1-500 vs 376K+ vs 3427+), meaning a direct row-level join is impossible.

**Strategy: Departmental Aggregation Join**
- `employee_performance_pro.csv` (500 rows) is the ML spine
- `Employee_Performance_Dataset.csv` (5000 rows) is joined via Department+JobRole aggregate averages
- `Cleaned_HR_Data_Analysis.csv` (2845 rows) is joined via Department+BusinessUnit aggregate averages

This is a real enterprise pattern: when individual IDs don't match, enrich with **group-level benchmarks**.

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/processed/features', exist_ok=True)

df_main = pd.read_csv('../data/raw/employee_performance_pro.csv')
df_perf = pd.read_csv('../data/raw/Employee_Performance_Dataset.csv')
df_hr   = pd.read_csv('../data/raw/Cleaned_HR_Data_Analysis.csv')

print(f'Main: {df_main.shape} | IDs: {df_main["EmployeeID"].min()}-{df_main["EmployeeID"].max()}')
print(f'Perf: {df_perf.shape} | IDs: {df_perf["Employee ID"].min()}-{df_perf["Employee ID"].max()}')
print(f'HR:   {df_hr.shape}  | IDs: {df_hr["Employee ID"].min()}-{df_hr["Employee ID"].max()}')
print()
print('IDs do NOT overlap - using departmental aggregation join instead of row-level join.')

Main: (500, 24) | IDs: 1-500
Perf: (5000, 13) | IDs: 100021-999957
HR:   (2845, 28)  | IDs: 1001-4000

IDs do NOT overlap - using departmental aggregation join instead of row-level join.


In [2]:
# --- Clean employee_performance_pro.csv ---
df = df_main.copy()
drop_cols = ['PhoneNumber', 'LeaveDayName', 'LastLeaveDate', 'CountryCode']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
df.rename(columns={'WorkLifeBalanceScore': 'WorkLifeBalance'}, inplace=True)
df['Attrition_encoded'] = (df['AttritionRisk'].str.strip().str.lower() == 'yes').astype(int)

# Fill missing CustomerSatisfaction with median
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].fillna('Unknown')

print(f'Cleaned main df: {df.shape}')
print(f'Missing values remaining: {df.isnull().sum().sum()}')
print(f'Attrition: {df["Attrition_encoded"].value_counts().to_dict()}')

Cleaned main df: (500, 21)
Missing values remaining: 0
Attrition: {0: 445, 1: 55}


In [3]:
# --- Departmental Aggregation Join ---
# Build dept+role averages from the performance dataset
df_perf_clean = df_perf.copy()
df_perf_clean.rename(columns={
    'Job Role': 'JobRole',
    'Performance Score': 'KPI_PerformanceScore',
    'KPI Score': 'KPI_Score',
    'Attendance (%)': 'Attendance_pct',
    'Peer Rating': 'Peer_Rating',
    'Task Completion (%)': 'TaskCompletion_pct',
    'Work Hours Logged': 'WorkHoursLogged',
    'Training Hours': 'TrainingHours_perf'
}, inplace=True)

perf_agg = df_perf_clean.groupby(['Department', 'JobRole']).agg(
    avg_KPI_Score=('KPI_Score', 'mean'),
    avg_Attendance_pct=('Attendance_pct', 'mean'),
    avg_Peer_Rating=('Peer_Rating', 'mean'),
    avg_TaskCompletion_pct=('TaskCompletion_pct', 'mean'),
    avg_WorkHoursLogged=('WorkHoursLogged', 'mean'),
).reset_index().round(2)

print(f'Performance aggregation table: {perf_agg.shape}')
print(perf_agg.head(3))

# Build dept averages from HR/engagement dataset
df_hr_clean = df_hr.copy()
df_hr_clean.rename(columns={
    'Engagement Score': 'EngagementScore',
    'Satisfaction Score': 'SatisfactionScore',
    'Work-Life Balance Score': 'WLB_Score_hr',
    'Current Employee Rating': 'EmployeeRating',
    'BusinessUnit': 'BusinessUnit',
    'DepartmentType': 'Department'
}, inplace=True)

hr_agg = df_hr_clean.groupby('Department').agg(
    avg_EngagementScore=('EngagementScore', 'mean'),
    avg_SatisfactionScore=('SatisfactionScore', 'mean'),
    avg_WLB_Score=('WLB_Score_hr', 'mean'),
).reset_index().round(2)

print(f'HR aggregation table: {hr_agg.shape}')
print(hr_agg.head(3))

Performance aggregation table: (15, 7)
  Department            JobRole  avg_KPI_Score  avg_Attendance_pct  \
0    Finance         Accountant          77.24               88.17   
1    Finance            Auditor          77.97               87.35   
2    Finance  Financial Analyst          76.97               87.09   

   avg_Peer_Rating  avg_TaskCompletion_pct  avg_WorkHoursLogged  
0             4.01                   85.22                45.10  
1             3.99                   84.69                44.61  
2             4.03                   84.89                45.34  
HR aggregation table: (6, 4)
         Department  avg_EngagementScore  avg_SatisfactionScore  avg_WLB_Score
0     Admin Offices                 2.95                   2.51           3.20
1  Executive Office                 3.38                   3.08           3.29
2             IT/IS                 3.02                   3.01           2.98


In [4]:
# Join aggregated benchmarks onto main df by Department+JobRole
merged = df.merge(perf_agg, on=['Department', 'JobRole'], how='left')

# HR agg joins on Department only (no JobRole in HR dataset)
merged = merged.merge(hr_agg, on='Department', how='left')

# Fill any remaining NaN (unmatched depts) with overall median
num_cols2 = merged.select_dtypes(include='number').columns
merged[num_cols2] = merged[num_cols2].fillna(merged[num_cols2].median())
str_cols2 = merged.select_dtypes(include='object').columns
merged[str_cols2] = merged[str_cols2].fillna('Unknown')

print(f'Final merged shape: {merged.shape}')
print(f'Missing values: {merged.isnull().sum().sum()}')  # must be 0!
print(f'Columns: {list(merged.columns)}')

merged.to_csv('../data/processed/employee_attrition_processed.csv', index=False)
merged[['EmployeeID', 'avg_EngagementScore']].rename(columns={'avg_EngagementScore': 'EngagementScore'}).to_csv('../data/processed/engagement_processed.csv', index=False)

for fname in ['essential_skills.csv', 'software_skills.csv', 'occupation_data.csv']:
    src = f'../data/raw/{fname}'
    if os.path.exists(src):
        pd.read_csv(src).to_csv(f'../data/processed/{fname.replace(".csv", "_processed.csv")}', index=False)

print('Saved all processed files.')
merged.head(3)

Final merged shape: (500, 29)
Missing values: 0
Columns: ['EmployeeID', 'Name', 'Gender', 'Age', 'Department', 'JobRole', 'EducationLevel', 'JoiningDate', 'Country', 'MonthlySalary', 'OvertimeHoursPerMonth', 'LeavesTaken', 'ProjectsHandled', 'TrainingHours', 'CustomerSatisfaction', 'LastPromotionYear', 'YearsAtCompany', 'WorkLifeBalance', 'PerformanceRating', 'AttritionRisk', 'Attrition_encoded', 'avg_KPI_Score', 'avg_Attendance_pct', 'avg_Peer_Rating', 'avg_TaskCompletion_pct', 'avg_WorkHoursLogged', 'avg_EngagementScore', 'avg_SatisfactionScore', 'avg_WLB_Score']


Saved all processed files.


,EmployeeID,Name,Gender,Age,Department,JobRole,EducationLevel,JoiningDate,Country,MonthlySalary,...,AttritionRisk,Attrition_encoded,avg_KPI_Score,avg_Attendance_pct,avg_Peer_Rating,avg_TaskCompletion_pct,avg_WorkHoursLogged,avg_EngagementScore,avg_SatisfactionScore,avg_WLB_Score
0,1,Steven Barnett,Other,57,Finance,Auditor,2,2016-05-05,India,93531,...,No,0,77.97,87.35,3.99,84.69,44.61,2.98,3.14,3.03
1,2,Christopher Benson,Female,26,Sales,Sales Executive,2,2014-08-20,Canada,53706,...,Yes,1,77.74,87.30,4.02,85.05,44.37,2.98,3.14,3.03
2,3,Norman Lane,Other,59,Support,Helpdesk,1,2010-05-17,Canada,105756,...,No,0,77.69,87.30,3.99,85.05,44.81,2.98,3.14,3.03
